# 03 – Model Training & Evaluation
**PhishGuard URL · EATC Assignment 2 · 2026**

This notebook:
1. Loads the preprocessed split saved by `02_preprocessing.ipynb`
2. Trains **5 candidate models** on the training split
3. Compares them on the **validation** split (primary metric: phishing F1)
4. Selects the winner, refits on train+val, evaluates **once** on the test split
5. Saves all deployment artifacts to `../models/` and reports to `../reports/`

The test split is **never** used for model selection.

In [1]:
import os, json, pickle, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, log_loss, brier_score_loss,
    confusion_matrix, balanced_accuracy_score, roc_curve, precision_recall_curve
)
from sklearn.inspection import permutation_importance

os.makedirs('../models', exist_ok=True)
os.makedirs('../reports', exist_ok=True)
print('Libraries loaded.')

Libraries loaded.


## 1. Load Split & Feature Names

In [2]:
with open('../models/train_val_test_split.pkl', 'rb') as f:
    train_df, val_df, test_df = pickle.load(f)

with open('../models/feature_names.pkl', 'rb') as f:
    FEATURE_NAMES = pickle.load(f)

X_train = train_df[FEATURE_NAMES].values;  y_train = train_df['label'].values
X_val   = val_df[FEATURE_NAMES].values;    y_val   = val_df['label'].values
X_test  = test_df[FEATURE_NAMES].values;   y_test  = test_df['label'].values

print(f'Train : {X_train.shape}  |  Val : {X_val.shape}  |  Test : {X_test.shape}')
print(f'Features: {FEATURE_NAMES}')

Train : (80489, 22)  |  Val : (17397, 22)  |  Test : (17344, 22)
Features: ['url_len', 'dom_len', 'is_ip', 'tld_len', 'subdom_cnt', 'letter_cnt', 'digit_cnt', 'special_cnt', 'eq_cnt', 'qm_cnt', 'amp_cnt', 'dot_cnt', 'dash_cnt', 'under_cnt', 'letter_ratio', 'digit_ratio', 'spec_ratio', 'is_https', 'slash_cnt', 'entropy', 'path_len', 'query_len']


## 2. Evaluation Helper

In [3]:
def evaluate(name, model, X, y):
    from sklearn.metrics import confusion_matrix
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]
    cm = confusion_matrix(y, y_pred)
    return {
        'model_name':      name,
        'accuracy':        float(accuracy_score(y, y_pred)),
        'phish_precision': float(precision_score(y, y_pred, pos_label=1, zero_division=0)),
        'phish_recall':    float(recall_score(y, y_pred, pos_label=1, zero_division=0)),
        'phish_f1':        float(f1_score(y, y_pred, pos_label=1, zero_division=0)),
        'weighted_f1':     float(f1_score(y, y_pred, average='weighted', zero_division=0)),
        'roc_auc':         float(roc_auc_score(y, y_prob)),
        'pr_auc':          float(average_precision_score(y, y_prob)),
        'log_loss':        float(log_loss(y, y_prob)),
        'false_positives': int(cm[0, 1]),
        'false_negatives': int(cm[1, 0]),
    }

print('evaluate() defined.')

evaluate() defined.


## 3. Train 5 Candidate Models on Training Split

### Model 1 — Logistic Regression

In [4]:
lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42,
        solver='lbfgs',
        n_jobs=-1,
    )),
])
lr.fit(X_train, y_train)
print('Logistic Regression trained.')

Logistic Regression trained.


### Model 2 — Decision Tree

In [5]:
dt = DecisionTreeClassifier(
    max_depth=18,
    min_samples_leaf=4,
    class_weight='balanced',
    random_state=42,
)
dt.fit(X_train, y_train)
print('Decision Tree trained.')

Decision Tree trained.


### Model 3 — Random Forest

In [6]:
rf = RandomForestClassifier(
    n_estimators=180,
    min_samples_leaf=2,
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
print('Random Forest trained.')

Random Forest trained.


### Model 4 — Histogram Gradient Boosting

In [7]:
hgb = HistGradientBoostingClassifier(
    max_iter=200,
    learning_rate=0.08,
    max_leaf_nodes=31,
    class_weight='balanced',
    random_state=42,
)
hgb.fit(X_train, y_train)
print('Histogram Gradient Boosting trained.')

  File "C:\Users\Lenovo\anaconda3\envs\phishguard\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\Lenovo\anaconda3\envs\phishguard\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Lenovo\anaconda3\envs\phishguard\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\Lenovo\anaconda3\envs\phishguard\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


Histogram Gradient Boosting trained.


### Model 5 — MLP Neural Network

In [8]:
mlp = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation='relu',
        solver='adam',
        early_stopping=True,
        learning_rate='adaptive',
        random_state=42,
        max_iter=300,
    )),
])
mlp.fit(X_train, y_train)
print('MLP Neural Network trained.')

MLP Neural Network trained.


## 4. Validate & Select Best Model (Phishing F1)

In [9]:
candidates = [
    ('Logistic Regression',         lr),
    ('Decision Tree',                dt),
    ('Random Forest',                rf),
    ('Histogram Gradient Boosting',  hgb),
    ('MLP Neural Network',           mlp),
]

val_results = [evaluate(name, model, X_val, y_val) for name, model in candidates]
val_df_results = pd.DataFrame(val_results).sort_values('phish_f1', ascending=False)

# Format for display
disp = val_df_results.copy()
for col in ['accuracy','phish_precision','phish_recall','phish_f1','weighted_f1']:
    disp[col] = disp[col].map(lambda x: f'{x*100:.2f}%')
for col in ['roc_auc','pr_auc']:
    disp[col] = disp[col].map(lambda x: f'{x:.4f}')
print(disp[['model_name','accuracy','phish_precision','phish_recall',
            'phish_f1','weighted_f1','roc_auc','pr_auc']].to_string(index=False))

                 model_name accuracy phish_precision phish_recall phish_f1 weighted_f1 roc_auc pr_auc
Histogram Gradient Boosting   97.32%          87.29%       95.14%   91.05%      97.37%  0.9930 0.9718
         MLP Neural Network   97.37%          90.29%       91.45%   90.87%      97.37%  0.9906 0.9662
              Random Forest   97.29%          92.19%       88.56%   90.34%      97.26%  0.9911 0.9614
              Decision Tree   94.90%          79.49%       86.79%   82.98%      94.99%  0.9272 0.8320
        Logistic Regression   92.86%          68.20%       93.86%   79.00%      93.30%  0.9742 0.8660


In [10]:
best_name = val_df_results.iloc[0]['model_name']
best_model_dict = dict(candidates)
best_model = best_model_dict[best_name]

print(f'\nSelected model: {best_name}  (highest validation phishing F1)')

val_df_results.to_csv('../reports/validation_comparison.csv', index=False)
print('Saved: reports/validation_comparison.csv')


Selected model: Histogram Gradient Boosting  (highest validation phishing F1)
Saved: reports/validation_comparison.csv


## 5. Refit Winner on Train + Validation

In [11]:
X_dev = np.vstack([X_train, X_val])
y_dev = np.concatenate([y_train, y_val])

# Use sklearn.base.clone to create a fresh estimator with the same hyperparameters.
# This avoids refitting on data the model already saw during validation comparison.
best_clf = clone(best_model)
best_clf.fit(X_dev, y_dev)
print(f'{best_name} cloned and refitted on {len(y_dev):,} development rows (train + val).')

Histogram Gradient Boosting cloned and refitted on 97,886 development rows (train + val).


## 6. Final One-Time Test Evaluation

**This cell is the only place test data is used.**

In [12]:
y_test_pred = best_clf.predict(X_test)
y_test_prob = best_clf.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_test_pred)
tn, fp, fn, tp = cm.ravel()

fpr_arr, tpr_arr, _ = roc_curve(y_test, y_test_prob)
prec_arr, rec_arr, _ = precision_recall_curve(y_test, y_test_prob)

total_rows = len(train_df) + len(val_df) + len(test_df)
leg_rows   = int((train_df['label'].eq(0).sum() + val_df['label'].eq(0).sum() + test_df['label'].eq(0).sum()))
phish_rows = int((train_df['label'].eq(1).sum() + val_df['label'].eq(1).sum() + test_df['label'].eq(1).sum()))

final_metrics = {
    'model_name':        best_name,
    'selection_metric':  'validation phishing F1',
    'train_rows':        int(len(y_train)),
    'val_rows':          int(len(y_val)),
    'dev_rows':          int(len(y_dev)),
    'test_rows':         int(len(y_test)),
    'total_rows':        int(total_rows),
    'legitimate_rows':   leg_rows,
    'phishing_rows':     phish_rows,
    'domain_overlap':    0,
    'random_state':      42,
    'split_strategy':    'StratifiedGroupKFold (20 folds; 14 train / 3 val / 3 test; grouped by registrable domain)',
    'feature_source':    'Recomputed from normalised URL text in preprocessing notebook',
    'accuracy':          float(accuracy_score(y_test, y_test_pred)),
    'balanced_accuracy': float(balanced_accuracy_score(y_test, y_test_pred)),
    'phish_precision':   float(precision_score(y_test, y_test_pred, pos_label=1, zero_division=0)),
    'phish_recall':      float(recall_score(y_test, y_test_pred, pos_label=1, zero_division=0)),
    'phish_f1':          float(f1_score(y_test, y_test_pred, pos_label=1, zero_division=0)),
    'weighted_f1':       float(f1_score(y_test, y_test_pred, average='weighted', zero_division=0)),
    'roc_auc':           float(roc_auc_score(y_test, y_test_prob)),
    'pr_auc':            float(average_precision_score(y_test, y_test_prob)),
    'log_loss':          float(log_loss(y_test, y_test_prob)),
    'brier_score':       float(brier_score_loss(y_test, y_test_prob)),
    'true_negatives':    int(tn),
    'false_positives':   int(fp),
    'false_negatives':   int(fn),
    'true_positives':    int(tp),
}

print(f"Accuracy:        {final_metrics['accuracy']*100:.2f}%")
print(f"Phishing F1:     {final_metrics['phish_f1']*100:.2f}%")
print(f"Phishing Recall: {final_metrics['phish_recall']*100:.2f}%")
print(f"ROC-AUC:         {final_metrics['roc_auc']:.4f}")
print(f"False Negatives: {fn:,}  (phishing URLs missed — highest security risk)")

Accuracy:        96.89%
Phishing F1:     89.48%
Phishing Recall: 92.21%
ROC-AUC:         0.9918
False Negatives: 194  (phishing URLs missed — highest security risk)


## 7. Permutation Feature Importance (Validation Split)

In [13]:
# Permutation importance is calculated using the train-only model on X_val.
# best_model was trained on X_train only and evaluated on X_val for model selection.
# This avoids leakage: best_clf has already seen val data (trained on train+val).
def _phish_f1(estimator, X, y):
    return f1_score(y, estimator.predict(X), pos_label=1, zero_division=0)

pi = permutation_importance(
    best_model, X_val, y_val,
    scoring=_phish_f1, n_repeats=10, random_state=42, n_jobs=-1
)
imp_df = pd.DataFrame({
    'feature':         FEATURE_NAMES,
    'importance_mean': pi.importances_mean,
    'importance_std':  pi.importances_std,
}).sort_values('importance_mean', ascending=False).reset_index(drop=True)

imp_df.to_csv('../reports/permutation_importance.csv', index=False)
print('Saved: reports/permutation_importance.csv')
print('Note: importance computed on train-only model (best_model) to avoid val-data leakage.')
print(imp_df.head(10).to_string(index=False))

Saved: reports/permutation_importance.csv
Note: importance computed on train-only model (best_model) to avoid val-data leakage.
    feature  importance_mean  importance_std
    dom_len         0.178314        0.002913
   path_len         0.123649        0.002113
   is_https         0.113024        0.003996
    tld_len         0.065984        0.002829
    url_len         0.062372        0.002527
    entropy         0.058289        0.002518
 subdom_cnt         0.055547        0.003498
   dash_cnt         0.023580        0.001946
    dot_cnt         0.022897        0.001570
special_cnt         0.015807        0.001004


## 8. Save All Artifacts

In [14]:
# --- models/ ---
with open('../models/best_model.pkl', 'wb') as f:
    pickle.dump(best_clf, f)
print('Saved: models/best_model.pkl')

with open('../models/best_model_name.pkl', 'wb') as f:
    pickle.dump(best_name, f)
print('Saved: models/best_model_name.pkl')

with open('../models/best_model_evaluation.json', 'w', encoding='utf-8') as f:
    json.dump(final_metrics, f, indent=2)
print('Saved: models/best_model_evaluation.json')

# --- reports/ ---
pd.DataFrame([final_metrics]).to_csv('../reports/final_test_metrics.csv', index=False)
print('Saved: reports/final_test_metrics.csv')

curves = {
    'label':     best_name,
    'fpr':       fpr_arr.tolist(),
    'tpr':       tpr_arr.tolist(),
    'precision': prec_arr.tolist(),
    'recall':    rec_arr.tolist(),
}
with open('../reports/test_curves.json', 'w') as f:
    json.dump(curves, f)
print('Saved: reports/test_curves.json')

Saved: models/best_model.pkl
Saved: models/best_model_name.pkl
Saved: models/best_model_evaluation.json
Saved: reports/final_test_metrics.csv
Saved: reports/test_curves.json


In [15]:
# Defanged test-set analysis for safe display
import re
analysis = test_df[['url', 'label']].copy()
analysis['phish_probability'] = y_test_prob
analysis['predicted_label']   = y_test_pred
analysis['display_url'] = (
    analysis['url']
    .str.replace(r'^https?://', lambda m: m.group().replace('http','hxxp'), regex=True)
    .str.replace('.', '[.]', regex=False)
    .str[:80]
)
analysis.drop(columns=['url']).to_csv('../reports/test_analysis.csv', index=False)
print('Saved: reports/test_analysis.csv')

Saved: reports/test_analysis.csv


In [16]:
# --- reports/full_test_set_results.csv ---
# Complete row-level test predictions (gitignored — too large to commit)
full_results = test_df[['url', 'label']].copy()
full_results['predicted_label']   = y_test_pred
full_results['phish_probability'] = y_test_prob
full_results.to_csv('../reports/full_test_set_results.csv', index=False)
print('Saved: reports/full_test_set_results.csv')

# --- reports/recommended_demo_cases.csv ---
# High-confidence correct predictions — useful for demo walkthrough
correct_phish = full_results[
    (full_results['label'] == 1) &
    (full_results['predicted_label'] == 1)
].nlargest(5, 'phish_probability')

correct_legit = full_results[
    (full_results['label'] == 0) &
    (full_results['predicted_label'] == 0)
].nsmallest(5, 'phish_probability')

demo_df = pd.concat([correct_phish, correct_legit]).reset_index(drop=True)
demo_df.to_csv('../reports/recommended_demo_cases.csv', index=False)
print('Saved: reports/recommended_demo_cases.csv')

# Defanged version — safe to commit (no live URLs)
safe_demo = demo_df.copy()
safe_demo['url'] = (
    safe_demo['url']
    .str.replace(r'^https?://', lambda m: m.group().replace('http', 'hxxp'), regex=True)
    .str.replace('.', '[.]', regex=False)
)
safe_demo.to_csv('../reports/recommended_demo_cases_defanged.csv', index=False)
print('Saved: reports/recommended_demo_cases_defanged.csv')
print()
print('Demo cases preview:')
print(safe_demo[['url','label','predicted_label','phish_probability']].to_string(index=False))

Saved: reports/full_test_set_results.csv
Saved: reports/recommended_demo_cases.csv
Saved: reports/recommended_demo_cases_defanged.csv

Demo cases preview:
                                                                                                                                                                                                                                                              url  label  predicted_label  phish_probability
                                                                                                                                                                                                                 hxxp://allegrolokalnie[.]pl-oferta1947202[.]cyou      1                1           0.999884
hxxps://dev[.]citoga[.]com/wp-content/plugins/wpforms-lite/vendor_prefixed/apimatic/core/src/Request/Parameters/index[.]php?r=bD1odHRwczovL3ZvbHVtZS1nNXA3Ymh4Zi1zdGF0aWMtb3JnLnMzLnVzLWVhc3QtMS5hbWF6b25hd3MuY29tL0tHbkhvT1Z2dU01QWU/ZW09aW5mb0Blb

## 9. Visualisations

In [17]:
# Bar chart — validation phishing F1
fig, ax = plt.subplots(figsize=(9, 5))
plot_data = val_df_results.sort_values('phish_f1')
ax.barh(plot_data['model_name'], plot_data['phish_f1'], color='#2563EB')
ax.set_xlabel('Validation phishing F1')
ax.set_title('Candidate Model Comparison')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1%}'))
plt.tight_layout()
plt.savefig('../reports/fig_model_comparison.png', dpi=150)
plt.close()
print('Saved: reports/fig_model_comparison.png')

Saved: reports/fig_model_comparison.png


In [18]:
# ROC and PR curves (test set)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(fpr_arr, tpr_arr, color='#2563EB', linewidth=2)
ax1.plot([0,1],[0,1],'k--',linewidth=1)
ax1.set_xlabel('False Positive Rate'); ax1.set_ylabel('True Positive Rate')
ax1.set_title(f"ROC (AUC = {final_metrics['roc_auc']:.4f})")
ax2.plot(rec_arr, prec_arr, color='#16a34a', linewidth=2)
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.set_title(f"Precision-Recall (AUC = {final_metrics['pr_auc']:.4f})")
plt.tight_layout()
plt.savefig('../reports/fig_test_curves.png', dpi=150)
plt.close()
print('Saved: reports/fig_test_curves.png')

Saved: reports/fig_test_curves.png


In [19]:
# Feature importance bar chart
top10 = imp_df.head(10)
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top10['feature'][::-1], top10['importance_mean'][::-1],
        xerr=top10['importance_std'][::-1], color='#2563EB')
ax.set_xlabel('Mean validation phishing F1 decrease')
ax.set_title('Top 10 Permutation Feature Importances')
plt.tight_layout()
plt.savefig('../reports/fig_feature_importance.png', dpi=150)
plt.close()
print('Saved: reports/fig_feature_importance.png')

Saved: reports/fig_feature_importance.png


## 10. Summary

All artifacts saved to `../models/` and `../reports/`.

Launch the Streamlit app:

```
streamlit run app/main.py
```